# LeetCode 1081 — Smallest Subsequence of Distinct Characters bằng Python

Link đề: https://leetcode.com/problems/smallest-subsequence-of-distinct-characters/ · Độ khó: **Medium** · Chủ đề: Stack (ngăn xếp đơn điệu), Greedy (tham lam), String (chuỗi)

Skeleton LeetCode:

```python
class Solution:
    def smallestSubsequence(self, s: str) -> str:
        ...
```

Ghi chú: cho chuỗi `s`, trả về **dãy con** (subsequence) **nhỏ nhất theo thứ tự từ điển** của `s` chứa **mọi ký tự khác nhau** của `s` **đúng một lần**. (Bài này giống hệt LeetCode 316 *Remove Duplicate Letters*.) Ý tưởng chính: dùng **ngăn xếp đơn điệu**: duyệt từng ký tự `c`; nếu `c` đã nằm trong ngăn xếp thì bỏ qua; ngược lại, chừng nào đỉnh ngăn xếp **lớn hơn `c`** và đỉnh đó **còn xuất hiện lại phía sau** thì bỏ đỉnh (vì sau này vẫn có thể đặt nó lại, và đặt `c` nhỏ hơn lên trước làm chuỗi nhỏ hơn), rồi đẩy `c` vào.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chuỗi kết quả là dãy con (giữ thứ tự tương đối) chứa mỗi ký tự khác nhau của `s` **đúng một lần**, và chọn chuỗi nhỏ nhất theo thứ tự từ điển.
2. Nắm các công cụ Python cần thiết: `dict` vị trí xuất hiện cuối cùng, `set` kiểm tra thành viên, `collections.Counter`, danh sách làm ngăn xếp, so sánh ký tự và chuỗi.
3. Hiểu **ngăn xếp đơn điệu có điều kiện**: chỉ bỏ đỉnh lớn hơn khi nó **còn xuất hiện sau này**.
4. Đi qua 4 cách: liệt kê mọi dãy con (brute force, dùng làm oracle), chọn tham lam từng vị trí bằng kiểm tra khả thi, ngăn xếp đơn điệu với vị trí cuối cùng (cách nộp chính) và ngăn xếp với bộ đếm số lần xuất hiện còn lại.
5. Tránh các lỗi kinh điển: sắp xếp các ký tự khác nhau, bỏ đỉnh mà không kiểm tra "còn xuất hiện sau", quên bỏ qua ký tự đã có trong ngăn xếp, chỉ giữ lần xuất hiện đầu tiên và chọn ký tự nhỏ nhất mà không xét khả thi.

## 2. Đề bài & ví dụ

Cho chuỗi `s`, trả về **dãy con nhỏ nhất theo thứ tự từ điển** của `s` chứa **tất cả ký tự khác nhau** của `s` **đúng một lần**.

**Ví dụ 1:** `s = "bcabc"` -> `"abc"`.

**Ví dụ 2:** `s = "cbacdcbc"` -> `"acdb"`.

**Ràng buộc:**

- `1 <= s.length <= 1000`
- `s` chỉ gồm các chữ cái tiếng Anh viết thường.

(Bài này giống LeetCode 316.)

In [1]:
from typing import List
from collections import Counter
from itertools import combinations, product
from functools import lru_cache
import random, string, time

examples = [
    ("bcabc", "abc"),
    ("cbacdcbc", "acdb"),
]
for s, expected in examples:
    print(f"s={s!r} -> kỳ vọng {expected!r}")

s='bcabc' -> kỳ vọng 'abc'
s='cbacdcbc' -> kỳ vọng 'acdb'


## 3. Kiến thức nền

### 3.1. Dãy con và thứ tự từ điển

Dãy con giữ nguyên **thứ tự tương đối**: `"acdb"` là dãy con của `"cbacdcbc"` (lấy `a` ở vị trí `2`, `c` ở `3`, `d` ở `4`, `b` ở `6`). So sánh chuỗi theo từ điển: từ trái sang phải, ký tự khác nhau đầu tiên quyết định; nếu một chuỗi là tiền tố của chuỗi kia thì chuỗi ngắn hơn nhỏ hơn.

In [2]:
print("acdb" < "acdbx", "abc" < "abd", "b" > "abc", sorted(["bca", "abc", "acb"]))
def is_subsequence(p, s):
    it = iter(s)
    return all(c in it for c in p)
print(is_subsequence("acdb", "cbacdcbc"), is_subsequence("abcd", "cbacdcbc"))

True True True ['abc', 'acb', 'bca']
True False


### 3.2. Vị trí xuất hiện cuối cùng

`last = {c: i for i, c in enumerate(s)}` ghi nhớ vị trí **cuối cùng** của mỗi ký tự (các phép gán sau ghi đè phép gán trước). Điều kiện "ký tự `x` còn xuất hiện sau vị trí `i`" chính là `last[x] > i`.

In [3]:
s = "cbacdcbc"
last = {c: i for i, c in enumerate(s)}
print(last)
print("b còn xuất hiện sau vị trí 3?", last["b"] > 3, "| a còn xuất hiện sau vị trí 3?", last["a"] > 3)

{'c': 7, 'b': 6, 'a': 2, 'd': 4}
b còn xuất hiện sau vị trí 3? True | a còn xuất hiện sau vị trí 3? False


### 3.3. Ngăn xếp (stack) đơn điệu

Ngăn xếp là danh sách dùng `append` (đẩy) và `pop` (lấy đỉnh `stack[-1]`). **Ngăn xếp đơn điệu** giữ các phần tử theo thứ tự tăng dần (hoặc giảm dần) bằng cách bỏ các đỉnh vi phạm thứ tự trước khi đẩy phần tử mới. Ở bài này ta muốn chuỗi **tăng dần càng nhiều càng tốt**, nhưng chỉ được bỏ đỉnh nếu còn cơ hội đặt lại nó ở phía sau.

In [4]:
def increasing_stack(arr):
    st = []
    for x in arr:
        while st and st[-1] > x:
            st.pop()
        st.append(x)
    return st
print(increasing_stack([3, 1, 4, 1, 5, 9, 2, 6]))

[1, 1, 2, 6]


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 2: `s = "cbacdcbc"`. Các ký tự khác nhau: `a, b, c, d`. Vị trí cuối: `a:2, b:6, c:7, d:4`.

Duyệt từng ký tự, ngăn xếp ban đầu rỗng:

- `c` (vị trí 0): đẩy ⇒ `[c]`.
- `b` (1): đỉnh `c > b` và `c` còn xuất hiện sau (vị trí cuối 7 > 1) ⇒ bỏ `c`; đẩy `b` ⇒ `[b]`.
- `a` (2): đỉnh `b > a`, `b` còn sau (6 > 2) ⇒ bỏ `b`; đẩy `a` ⇒ `[a]`.
- `c` (3): đỉnh `a < c` ⇒ đẩy ⇒ `[a, c]`.
- `d` (4): đỉnh `c < d` ⇒ đẩy ⇒ `[a, c, d]`.
- `c` (5): đã có trong ngăn xếp ⇒ bỏ qua.
- `b` (6): đỉnh `d > b`, nhưng `d` **không còn** xuất hiện sau (vị trí cuối 4 < 6) ⇒ **không** được bỏ; `b` chưa có trong ngăn xếp ⇒ đẩy ⇒ `[a, c, d, b]`.
- `c` (7): đã có ⇒ bỏ qua.

Kết quả `"acdb"` ✓.

## 5. Cách 1 — Brute Force: liệt kê mọi dãy con

**Ý tưởng.** Với `n` ký tự có `2^n` dãy con (chọn tập vị trí). Giữ các dãy con mà **mỗi ký tự khác nhau của `s` xuất hiện đúng một lần**, rồi lấy dãy nhỏ nhất theo thứ tự từ điển. Đúng nguyên văn đề, không dùng tham lam nên là **oracle độc lập** (dùng được khi `n <= 16`).

In [5]:
def smallest_brute(s: str) -> str:
    n = len(s)
    distinct = sorted(set(s))
    best = None
    for mask in range(1 << n):
        chosen = [s[i] for i in range(n) if mask >> i & 1]
        if len(chosen) == len(distinct) and sorted(chosen) == distinct:   # mỗi ký tự khác nhau đúng một lần
            cand = "".join(chosen)
            if best is None or cand < best:
                best = cand
    return best

for s, expected in examples[:1]:
    got = smallest_brute(s)
    print(f"brute({s!r}) = {got!r}", "OK" if got == expected else "SAI")
    assert got == expected
assert smallest_brute("cbacdcbc") == "acdb"

brute('bcabc') = 'abc' OK


**Độ phức tạp và nút thắt.** `O(2^n * n)`: chỉ dùng được với `n` nhỏ. Đo thử:

In [6]:
rng = random.Random(1081)
for n in (10, 12, 14, 16):
    s = "".join(rng.choice("abcd") for _ in range(n))
    t0 = time.perf_counter()
    smallest_brute(s)
    dt = time.perf_counter() - t0
    print(f"n={n:2}: {dt * 1000:9.1f} ms")

n=10:       0.7 ms
n=12:       3.2 ms
n=14:      16.5 ms
n=16:      85.1 ms


Nút thắt: ta xét cả những tập vị trí không thể là đáp án. Có thể **chọn từng ký tự của đáp án** từ trái sang phải một cách tham lam.

## 6. Ý tưởng thuật toán / bước đột phá

**Chọn tham lam từng vị trí.** Đáp án có đúng `k` ký tự (số ký tự khác nhau). Để chuỗi nhỏ nhất, ký tự **đầu tiên** của đáp án phải nhỏ nhất có thể. Một ký tự `c` đặt ở đầu (tại lần xuất hiện **đầu tiên** của nó, vị trí `i`) là **khả thi** nếu mọi ký tự khác nhau của `s` còn xuất hiện trong `s[i:]` (để các ký tự còn lại vẫn có chỗ). Chọn `c` nhỏ nhất khả thi, rồi lặp lại cho phần `s[i + 1:]` với các ký tự còn thiếu. Cách 2 làm đúng việc này, `O(26 * n)`.

**Ngăn xếp đơn điệu (`O(n)`).** Làm cùng việc nhưng trong một lượt: duyệt từng ký tự `c`. Ngăn xếp giữ đáp án hiện tại (đang tăng dần theo thứ tự có thể). Nếu `c` đã nằm trong ngăn xếp thì bỏ qua (đã có phiên bản ở vị trí tốt hơn hoặc ngang bằng). Ngược lại, chừng nào đỉnh `t > c` **và** `t` còn xuất hiện lại sau vị trí hiện tại thì bỏ `t` (sau này có thể đặt lại `t`, trong khi `c < t` đứng trước làm chuỗi nhỏ hơn).

| Cách | Ý tưởng |
|---|---|
| 2 | Chọn từng ký tự của đáp án: ký tự nhỏ nhất mà phần còn lại vẫn chứa đủ các ký tự cần thiết |
| 3 | Ngăn xếp đơn điệu + vị trí xuất hiện cuối cùng (nộp) |
| 4 | Ngăn xếp đơn điệu + bộ đếm số lần xuất hiện còn lại |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Chọn tham lam từng ký tự của đáp án

**Giải thích từng đoạn.**

- `need` là tập các ký tự khác nhau còn thiếu; `start` là vị trí bắt đầu được phép chọn.
- Xét các ký tự của `need` theo thứ tự tăng dần; với ký tự `c`, `i` là lần xuất hiện đầu tiên của `c` tại hoặc sau `start`.
- `c` khả thi nếu mọi ký tự trong `need` đều có trong `s[i:]`. Chọn `c` khả thi nhỏ nhất, thêm vào đáp án, bỏ khỏi `need`, đặt `start = i + 1`.

In [7]:
def smallest_greedy(s: str) -> str:
    need = set(s)
    start = 0
    res = []
    while need:
        for c in sorted(need):
            i = s.index(c, start)                       # lần xuất hiện đầu tiên của c từ start trở đi
            if need <= set(s[i:]):                      # mọi ký tự còn thiếu vẫn xuất hiện từ i trở đi
                res.append(c)
                need.remove(c)
                start = i + 1
                break
    return "".join(res)

for s, expected in examples:
    got = smallest_greedy(s)
    print(f"tham lam({s!r}) = {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

tham lam('bcabc') = 'abc' OK
tham lam('cbacdcbc') = 'acdb' OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def smallestSubsequence(self, s: str) -> str:
        need = set(s)                                    # các ký tự khác nhau còn phải chọn
        start = 0                                        # chỉ được chọn từ vị trí này trở đi
        res = []
        while need:
            for c in sorted(need):                       # thử ký tự nhỏ nhất trước
                i = s.index(c, start)                    # lần xuất hiện đầu tiên của c từ start
                if need <= set(s[i:]):                   # các ký tự còn lại vẫn có chỗ phía sau
                    res.append(c)
                    need.remove(c)
                    start = i + 1
                    break
        return "".join(res)

sol_greedy = lambda s: Solution().smallestSubsequence(s)
for s, expected in examples:
    assert sol_greedy(s) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** có tối đa `26` vòng ngoài; mỗi vòng thử tối đa `26` ký tự, mỗi lần tạo `set(s[i:])` tốn `O(n)`: tổng `O(26^2 * n)` trong trường hợp xấu (thực tế nhỏ hơn nhiều). Bộ nhớ `O(n)`.

---

### 7.2. Cách 3 — Ngăn xếp đơn điệu với vị trí cuối cùng (cách nộp chính)

**Giải thích từng đoạn.**

- `last[c]`: vị trí xuất hiện cuối cùng của `c`.
- `stack`: các ký tự đang giữ; `in_stack`: tập các ký tự trong ngăn xếp (để kiểm tra `O(1)`).
- Với ký tự `c` ở vị trí `i`:
  - nếu `c in in_stack`: bỏ qua;
  - ngược lại: `while stack and stack[-1] > c and last[stack[-1]] > i`: bỏ đỉnh (`pop`, xóa khỏi `in_stack`); rồi `append(c)` và thêm vào `in_stack`.
- Kết quả là `"".join(stack)`.

In [9]:
def smallest_stack(s: str) -> str:
    last = {c: i for i, c in enumerate(s)}
    stack, in_stack = [], set()
    for i, c in enumerate(s):
        if c in in_stack:
            continue
        while stack and stack[-1] > c and last[stack[-1]] > i:
            in_stack.remove(stack.pop())
        stack.append(c)
        in_stack.add(c)
    return "".join(stack)

for s, expected in examples:
    got = smallest_stack(s)
    print(f"ngăn xếp({s!r}) = {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

ngăn xếp('bcabc') = 'abc' OK
ngăn xếp('cbacdcbc') = 'acdb' OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def smallestSubsequence(self, s: str) -> str:
        last = {c: i for i, c in enumerate(s)}           # vị trí xuất hiện cuối cùng của từng ký tự
        stack = []                                       # đáp án đang xây dựng
        in_stack = set()                                 # các ký tự đã có trong ngăn xếp
        for i, c in enumerate(s):
            if c in in_stack:                            # đã có c ở vị trí tốt: bỏ qua
                continue
            while stack and stack[-1] > c and last[stack[-1]] > i:   # đỉnh lớn hơn c và còn xuất hiện sau: bỏ đỉnh
                in_stack.remove(stack.pop())
            stack.append(c)
            in_stack.add(c)
        return "".join(stack)

sol = lambda s: Solution().smallestSubsequence(s)
for s, expected in examples:
    assert sol(s) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** mỗi ký tự được đẩy và bỏ tối đa một lần ⇒ thời gian `O(n)`, bộ nhớ `O(26)` (ngăn xếp tối đa `26` ký tự) cộng `O(26)` cho `last`.

---

### 7.3. Cách 4 — Ngăn xếp đơn điệu với bộ đếm còn lại

**Ý tưởng khác.** Thay vì lưu vị trí cuối cùng, đếm số lần mỗi ký tự **còn xuất hiện sau vị trí hiện tại**: `count[c]` ban đầu là tổng số lần; khi xử lý ký tự `c` ta giảm `count[c]` (ký tự này đã dùng). Điều kiện "đỉnh `t` còn xuất hiện sau" trở thành `count[t] > 0`.

In [11]:
def smallest_counter(s: str) -> str:
    count = Counter(s)
    stack, in_stack = [], set()
    for c in s:
        count[c] -= 1                                    # còn lại sau vị trí hiện tại
        if c in in_stack:
            continue
        while stack and stack[-1] > c and count[stack[-1]] > 0:
            in_stack.remove(stack.pop())
        stack.append(c)
        in_stack.add(c)
    return "".join(stack)

for s, expected in examples:
    got = smallest_counter(s)
    print(f"bộ đếm({s!r}) = {got!r}", "OK" if got == expected else "SAI")
    assert got == expected

bộ đếm('bcabc') = 'abc' OK
bộ đếm('cbacdcbc') = 'acdb' OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def smallestSubsequence(self, s: str) -> str:
        from collections import Counter
        remain = Counter(s)                              # remain[c]: số lần c còn xuất hiện (kể cả vị trí hiện tại lúc đầu)
        stack, in_stack = [], set()
        for c in s:
            remain[c] -= 1                               # từ đây remain[c] là số lần c xuất hiện SAU vị trí hiện tại
            if c in in_stack:
                continue
            while stack and stack[-1] > c and remain[stack[-1]] > 0:
                in_stack.remove(stack.pop())
            stack.append(c)
            in_stack.add(c)
        return "".join(stack)

sol_counter = lambda s: Solution().smallestSubsequence(s)
for s, expected in examples:
    assert sol_counter(s) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** thời gian `O(n)`, bộ nhớ `O(26)`.

## 8. TRACE từng bước

### 8.1. Ngăn xếp đơn điệu cho `"cbacdcbc"`

In từng ký tự, quyết định bỏ qua / bỏ đỉnh / đẩy và trạng thái ngăn xếp.

In [13]:
def trace_stack(s):
    last = {c: i for i, c in enumerate(s)}
    print("last =", last)
    stack, in_stack = [], set()
    for i, c in enumerate(s):
        if c in in_stack:
            print(f"i={i} c={c!r}: đã có trong ngăn xếp -> bỏ qua; stack = {stack}")
            continue
        while stack and stack[-1] > c and last[stack[-1]] > i:
            top = stack.pop()
            in_stack.remove(top)
            print(f"i={i} c={c!r}: đỉnh {top!r} > {c!r} và còn xuất hiện sau (last={last[top]} > {i}) -> bỏ đỉnh; stack = {stack}")
        if stack and stack[-1] > c:
            print(f"i={i} c={c!r}: đỉnh {stack[-1]!r} > {c!r} nhưng KHÔNG còn xuất hiện sau (last={last[stack[-1]]} <= {i}) -> giữ")
        stack.append(c)
        in_stack.add(c)
        print(f"i={i} c={c!r}: đẩy -> stack = {stack}")
    return "".join(stack)

assert trace_stack("cbacdcbc") == "acdb"

last = {'c': 7, 'b': 6, 'a': 2, 'd': 4}
i=0 c='c': đẩy -> stack = ['c']
i=1 c='b': đỉnh 'c' > 'b' và còn xuất hiện sau (last=7 > 1) -> bỏ đỉnh; stack = []
i=1 c='b': đẩy -> stack = ['b']
i=2 c='a': đỉnh 'b' > 'a' và còn xuất hiện sau (last=6 > 2) -> bỏ đỉnh; stack = []
i=2 c='a': đẩy -> stack = ['a']
i=3 c='c': đẩy -> stack = ['a', 'c']
i=4 c='d': đẩy -> stack = ['a', 'c', 'd']
i=5 c='c': đã có trong ngăn xếp -> bỏ qua; stack = ['a', 'c', 'd']
i=6 c='b': đỉnh 'd' > 'b' nhưng KHÔNG còn xuất hiện sau (last=4 <= 6) -> giữ
i=6 c='b': đẩy -> stack = ['a', 'c', 'd', 'b']
i=7 c='c': đã có trong ngăn xếp -> bỏ qua; stack = ['a', 'c', 'd', 'b']


### 8.2. Chọn tham lam từng ký tự cho `"bcabc"`

In ký tự được thử, vị trí và kiểm tra khả thi.

In [14]:
s = "bcabc"
need, start, res = set(s), 0, []
while need:
    for c in sorted(need):
        i = s.index(c, start)
        feasible = need <= set(s[i:])
        print(f"need={sorted(need)} start={start}: thử {c!r} tại vị trí {i}; s[{i}:]={s[i:]!r} chứa đủ? {feasible}")
        if feasible:
            res.append(c)
            need.remove(c)
            start = i + 1
            break
print("kết quả:", "".join(res))
assert "".join(res) == "abc"

need=['a', 'b', 'c'] start=0: thử 'a' tại vị trí 2; s[2:]='abc' chứa đủ? True
need=['b', 'c'] start=3: thử 'b' tại vị trí 3; s[3:]='bc' chứa đủ? True
need=['c'] start=4: thử 'c' tại vị trí 4; s[4:]='c' chứa đủ? True
kết quả: abc


### 8.3. Trường hợp không được bỏ đỉnh

`s = "bcab"`: khi gặp `a` ở vị trí 2, đỉnh `c > a` nhưng `c` không còn xuất hiện sau nên phải giữ (đáp án `"bca"`).

In [15]:
print(trace_stack("bcab"))
assert smallest_stack("bcab") == "bca" == smallest_brute("bcab")

last = {'b': 3, 'c': 1, 'a': 2}
i=0 c='b': đẩy -> stack = ['b']
i=1 c='c': đẩy -> stack = ['b', 'c']
i=2 c='a': đỉnh 'c' > 'a' nhưng KHÔNG còn xuất hiện sau (last=1 <= 2) -> giữ
i=2 c='a': đẩy -> stack = ['b', 'c', 'a']
i=3 c='b': đã có trong ngăn xếp -> bỏ qua; stack = ['b', 'c', 'a']
bca


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (ngăn xếp đơn điệu với vị trí cuối cùng): `O(n)` thời gian.

```python
class Solution:
    def smallestSubsequence(self, s: str) -> str:
        last = {c: i for i, c in enumerate(s)}           # vị trí xuất hiện cuối cùng của từng ký tự
        stack = []                                       # đáp án đang xây dựng
        in_stack = set()                                 # các ký tự đã có trong ngăn xếp
        for i, c in enumerate(s):
            if c in in_stack:                            # đã có c ở vị trí tốt: bỏ qua
                continue
            while stack and stack[-1] > c and last[stack[-1]] > i:   # đỉnh lớn hơn c và còn xuất hiện sau: bỏ đỉnh
                in_stack.remove(stack.pop())
            stack.append(c)
            in_stack.add(c)
        return "".join(stack)
```

In [16]:
class Solution:
    def smallestSubsequence(self, s: str) -> str:
        last = {c: i for i, c in enumerate(s)}           # vị trí xuất hiện cuối cùng của từng ký tự
        stack = []                                       # đáp án đang xây dựng
        in_stack = set()                                 # các ký tự đã có trong ngăn xếp
        for i, c in enumerate(s):
            if c in in_stack:                            # đã có c ở vị trí tốt: bỏ qua
                continue
            while stack and stack[-1] > c and last[stack[-1]] > i:   # đỉnh lớn hơn c và còn xuất hiện sau: bỏ đỉnh
                in_stack.remove(stack.pop())
            stack.append(c)
            in_stack.add(c)
        return "".join(stack)

sol = lambda s: Solution().smallestSubsequence(s)
for s, expected in examples:
    got = sol(s)
    print(repr(got), "OK" if got == expected else "SAI")
    assert got == expected

'abc' OK
'acdb' OK


## 10. Vì sao đúng

Gọi `k` là số ký tự khác nhau của `s`. Đáp án là dãy con dài `k` gồm mỗi ký tự khác nhau đúng một lần.

**Cách chọn tham lam (Cách 2) đúng.** Xét ký tự đầu của đáp án. Một ký tự `c` có thể đứng đầu khi và chỉ khi tồn tại một dãy con hợp lệ bắt đầu bằng `c`; dãy con đó có thể chọn `c` ở lần xuất hiện **đầu tiên** (dời `c` về lần xuất hiện sớm hơn chỉ làm các ký tự còn lại có thêm chỗ), nên điều kiện là: mọi ký tự khác nhau (trong `need`) đều xuất hiện trong `s[i:]` với `i` là lần xuất hiện đầu của `c`. Trong các `c` khả thi, chọn `c` **nhỏ nhất** cho chuỗi nhỏ nhất (so sánh từ điển ưu tiên ký tự đầu), và lần xuất hiện đầu tiên là tốt nhất cho phần còn lại (phần còn lại là một dãy con của `s[i + 1:]`, nên `i` nhỏ hơn cho nhiều lựa chọn hơn). Quy nạp cho phần còn lại với tập `need` và `start` mới.

**Ngăn xếp đơn điệu (Cách 3) đúng.** Bất biến: sau khi xử lý các ký tự `s[0..i]`, ngăn xếp là dãy con nhỏ nhất theo từ điển của `s[0..i]` mà chứa đúng một lần mọi ký tự đã xuất hiện trong `s[0..i]` **và** có thể mở rộng thành đáp án hợp lệ cho toàn bộ `s` (các ký tự bị bỏ đều còn xuất hiện sau). Khi đọc `c`:

- Nếu `c` đã trong ngăn xếp: đặt `c` lần nữa chỉ làm chuỗi dài hơn hoặc vi phạm "đúng một lần"; ngăn xếp hiện tại đã nhỏ nhất ⇒ giữ nguyên.
- Nếu không: ta cần chèn `c` vào cuối. Đỉnh `t > c` mà `t` còn xuất hiện sau (nên còn có thể chọn lại) thì bỏ `t` rồi để `c` đứng trước làm chuỗi nhỏ hơn (tại vị trí của `t`, ký tự `c < t`); nếu `t` **không** còn xuất hiện sau thì không được bỏ vì mất `t` khỏi đáp án. Dừng bỏ khi đỉnh `< c` hoặc không còn xuất hiện sau (bỏ thêm các phần tử phía dưới cũng không giúp nếu đỉnh này phải giữ, vì mọi thay đổi phải đi qua đỉnh).

**Cách 4** chỉ thay `last[t] > i` bằng `remain[t] > 0` (`remain[t]` là số lần `t` xuất hiện **sau** vị trí hiện tại), hai điều kiện tương đương.

**Kiểm chứng bằng máy.** Oracle (Cách 1) liệt kê mọi dãy con. Ta thêm một oracle thứ hai độc lập, **quy hoạch động theo (vị trí, tập ký tự đã dùng)** có nhớ đệ quy, dùng cho chuỗi dài hơn; rồi đối chiếu mọi cách trên mọi chuỗi ngắn trên bảng chữ cái nhỏ và nhiều chuỗi ngẫu nhiên.

In [17]:
def smallest_dp(s: str) -> str:
    """Oracle thứ hai: f(i, mask) = chuỗi nhỏ nhất từ s[i:] khi các ký tự trong mask đã dùng. Mọi lời giải hợp lệ cùng độ dài nên
    so sánh từ điển cho phép chọn tối ưu từng bước."""
    letters = sorted(set(s))
    bit = {c: 1 << k for k, c in enumerate(letters)}
    full = (1 << len(letters)) - 1
    n = len(s)

    @lru_cache(maxsize=None)
    def f(i, mask):
        if mask == full:
            return ""
        if i == n:
            return None
        best = f(i + 1, mask)                             # bỏ s[i]
        c = s[i]
        if not mask & bit[c]:                             # lấy s[i]
            rest = f(i + 1, mask | bit[c])
            if rest is not None and (best is None or c + rest < best):
                best = c + rest
        return best

    result = f(0, 0)
    f.cache_clear()
    return result

alphabet_tests = 0
for letters, max_len in (("abc", 8), ("abcd", 6)):
    for L in range(1, max_len + 1):
        for tup in product(letters, repeat=L):
            s = "".join(tup)
            ref = smallest_brute(s)
            assert smallest_dp(s) == ref, s
            for name, fn in (("tham lam", smallest_greedy), ("ngăn xếp", smallest_stack), ("bộ đếm", smallest_counter),
                             ("nộp (Cách 3)", sol), ("Cách 2 (nộp)", sol_greedy), ("Cách 4 (nộp)", sol_counter)):
                assert fn(s) == ref, (name, s, fn(s), ref)
            alphabet_tests += 1
print("đã đối chiếu", alphabet_tests, "chuỗi (bảng chữ {a,b,c} độ dài <= 8 và {a,b,c,d} độ dài <= 6): mọi cách khớp oracle")

đã đối chiếu 15300 chuỗi (bảng chữ {a,b,c} độ dài <= 8 và {a,b,c,d} độ dài <= 6): mọi cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê dãy con | `O(2^n * n)` | `O(n)` |
| 2. Chọn tham lam từng ký tự | `O(26^2 * n)` | `O(n)` |
| 3. Ngăn xếp + vị trí cuối | `O(n)` | `O(26)` |
| 4. Ngăn xếp + bộ đếm | `O(n)` | `O(26)` |

Đo ở cỡ tối đa (`n = 1000`) với các dạng: ngẫu nhiên 26 chữ, ngẫu nhiên ít chữ, giảm dần lặp, tăng dần lặp, toàn một chữ:

In [18]:
rng = random.Random(11)
rand = lambda letters, n=1000: "".join(rng.choice(letters) for _ in range(n))
cases = {
    "ngẫu nhiên 26 chữ": rand(string.ascii_lowercase),
    "ngẫu nhiên 3 chữ": rand("abc"),
    "giảm dần lặp (zyx...a)*": (string.ascii_lowercase[::-1] * 40)[:1000],
    "tăng dần lặp (abc...z)*": (string.ascii_lowercase * 40)[:1000],
    "toàn một chữ": "a" * 1000,
    "ba khối: b*, a*, c*": "b" * 300 + "a" * 300 + "c" * 400,
}
for label, s in cases.items():
    ref = sol(s)
    row = [f"{label:28}"]
    for name, fn in (("ngăn xếp", sol), ("bộ đếm", sol_counter), ("tham lam", sol_greedy)):
        t0 = time.perf_counter()
        got = fn(s)
        dt = (time.perf_counter() - t0) * 1000
        assert got == ref, (label, name)
        row.append(f"{name} {dt:7.3f} ms")
    print(" | ".join(row), "| kết quả", ref[:12] + ("..." if len(ref) > 12 else ""))

ngẫu nhiên 26 chữ            | ngăn xếp   0.155 ms | bộ đếm   0.127 ms | tham lam   0.180 ms | kết quả abcdefghijkl...
ngẫu nhiên 3 chữ             | ngăn xếp   0.058 ms | bộ đếm   0.056 ms | tham lam   0.061 ms | kết quả abc
giảm dần lặp (zyx...a)*      | ngăn xếp   0.132 ms | bộ đếm   0.147 ms | tham lam   0.116 ms | kết quả abcdefghijkl...
tăng dần lặp (abc...z)*      | ngăn xếp   0.077 ms | bộ đếm   0.085 ms | tham lam   0.059 ms | kết quả abcdefghijkl...
toàn một chữ                 | ngăn xếp   0.057 ms | bộ đếm   0.055 ms | tham lam   0.055 ms | kết quả a
ba khối: b*, a*, c*          | ngăn xếp   0.056 ms | bộ đếm   0.087 ms | tham lam   0.057 ms | kết quả bac


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), kiểm tra hợp lệ độc lập, test ngẫu nhiên có seed (đúng ràng buộc) và các tính chất.

In [19]:
all_impls = {"tham lam": smallest_greedy, "ngăn xếp": smallest_stack, "bộ đếm": smallest_counter,
             "nộp (Cách 3)": sol, "Cách 2 (nộp)": sol_greedy, "Cách 4 (nộp)": sol_counter}

def is_valid_result(s: str, res: str) -> bool:
    """Hợp lệ (không nhất thiết nhỏ nhất): res là dãy con của s chứa mỗi ký tự khác nhau đúng một lần."""
    return sorted(res) == sorted(set(s)) and is_subsequence(res, s)

def check_all(s, expected=None, brute=True):
    results = {name: fn(s) for name, fn in all_impls.items()}
    first = results["nộp (Cách 3)"]
    for name, got in results.items():
        assert got == first, (name, s[:30], got, first)
    assert is_valid_result(s, first), (s, first)
    if brute and len(s) <= 14:
        assert smallest_brute(s) == first, s
    if brute and len(set(s)) <= 10 and len(s) <= 120:
        assert smallest_dp(s) == first, s
    if expected is not None:
        assert first == expected, (s, first, expected)
    return first

# 12.1 ví dụ đề
for s, e in examples:
    check_all(s, e)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ("a", "a"),                                      # một ký tự
    ("aaaa", "a"),                                   # toàn một chữ: giữ một ký tự
    ("ba", "ba"),                                    # không thể đổi thứ tự (dãy con giữ thứ tự)
    ("abc", "abc"),                                  # đã tăng dần
    ("cba", "cba"),                                  # giảm dần, mỗi chữ chỉ xuất hiện một lần: không bỏ được
    ("bcabc", "abc"),                                # ví dụ 1
    ("cbacdcbc", "acdb"),                            # ví dụ 2
    ("abacb", "abc"),                                # đỉnh c không còn xuất hiện sau nên phải giữ
    ("bbcaac", "bac"),                               # kiểm tra bỏ qua ký tự đã trong ngăn xếp
    ("ecbacba", "eacb"),                             # ví dụ kinh điển của bài 316
    ("leetcode", "letcod"),                          # ví dụ kinh điển
    ("cdadabcc", "adbc"),                            # ví dụ kinh điển
    ("zzzabc", "zabc"),                              # z chỉ có ở đầu nên phải đứng đầu
    ("abcabc", "abc"),                               # lặp lại cả dãy
    ("cbacb", "acb"),                                # sau a: c rồi b
    ("ab" * 500, "ab"),                              # độ dài tối đa, hai chữ
    ("a" * 1000, "a"),                               # độ dài tối đa, một chữ
    (string.ascii_lowercase[::-1], string.ascii_lowercase[::-1]),      # 26 chữ giảm dần, mỗi chữ một lần
    (string.ascii_lowercase * 3, string.ascii_lowercase),              # 26 chữ lặp ba lần
]
for s, e in edge:
    check_all(s, e)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 19 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm so với oracle)
def wrong_sorted_distinct(s):                      # sắp xếp các ký tự khác nhau (bỏ qua yêu cầu dãy con giữ thứ tự)
    return "".join(sorted(set(s)))

def wrong_pop_always(s):                           # bỏ đỉnh lớn hơn c mà không kiểm tra "còn xuất hiện sau"
    stack, in_stack = [], set()
    for c in s:
        if c in in_stack:
            continue
        while stack and stack[-1] > c:
            in_stack.remove(stack.pop())
        stack.append(c)
        in_stack.add(c)
    return "".join(stack)

def wrong_no_skip(s):                              # quên bỏ qua ký tự đã có trong ngăn xếp (sinh trùng lặp)
    last = {c: i for i, c in enumerate(s)}
    stack = []
    for i, c in enumerate(s):
        while stack and stack[-1] > c and last[stack[-1]] > i:
            stack.pop()
        stack.append(c)
    return "".join(stack)

def wrong_first_occurrence(s):                     # chỉ giữ lần xuất hiện đầu tiên của mỗi ký tự
    seen, out = set(), []
    for c in s:
        if c not in seen:
            seen.add(c)
            out.append(c)
    return "".join(out)

def wrong_smallest_first(s):                       # chọn ký tự nhỏ nhất còn lại mà không xét khả thi
    need, start, res = set(s), 0, []
    while need:
        c = min(need)
        i = s.find(c, start)
        if i == -1:                                # không còn: bỏ qua (kết quả thiếu ký tự)
            need.remove(c)
            continue
        res.append(c)
        need.remove(c)
        start = i + 1
    return "".join(res)

def find_counter(wrong, tries=40000, seed=9):
    rng = random.Random(seed)
    best = None
    for _ in range(tries):
        s = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 8)))
        if best is not None and len(s) >= len(best):
            continue
        if wrong(s) != smallest_dp(s):
            best = s
    return best

hand_cases = {                                       # phản ví dụ chọn tay (dễ nhìn), được xác nhận lại bằng oracle
    "sắp xếp các ký tự khác nhau": (wrong_sorted_distinct, "cba"),                  # "abc" không phải dãy con của "cba"
    "bỏ đỉnh mà không kiểm tra còn xuất hiện sau": (wrong_pop_always, "ba"),        # b chỉ xuất hiện một lần nhưng bị bỏ
    "quên bỏ qua ký tự đã có": (wrong_no_skip, "cbacdcbc"),                         # ví dụ 2: sinh trùng
    "chỉ giữ lần xuất hiện đầu tiên": (wrong_first_occurrence, "bcabc"),            # "bca" thay vì "abc"
    "chọn ký tự nhỏ nhất mà không xét khả thi": (wrong_smallest_first, "cbacdcbc"), # ví dụ 2: mất d
}
counters = {}
for name, (fn, s) in hand_cases.items():
    wrong_result = fn(s)
    right = smallest_dp(s)
    assert wrong_result != right, name
    assert find_counter(fn) is not None, name           # tìm kiếm tự động cũng tìm ra phản ví dụ (nhỏ nhất)
    counters[name] = s
    print(f"{name:44} s={s!r}: sai -> {wrong_result!r}, đúng -> {right!r}")
    check_all(s)
print("12.3 phản ví dụ: OK")

sắp xếp các ký tự khác nhau                  s='cba': sai -> 'abc', đúng -> 'cba'


bỏ đỉnh mà không kiểm tra còn xuất hiện sau  s='ba': sai -> 'a', đúng -> 'ba'


quên bỏ qua ký tự đã có                      s='cbacdcbc': sai -> 'acdbc', đúng -> 'acdb'


chỉ giữ lần xuất hiện đầu tiên               s='bcabc': sai -> 'bca', đúng -> 'abc'


chọn ký tự nhỏ nhất mà không xét khả thi     s='cbacdcbc': sai -> 'abc', đúng -> 'acdb'
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên nhỏ: nhiều bảng chữ cái
rng = random.Random(4081)
for _ in range(3000):
    letters = string.ascii_lowercase[:rng.choice([1, 2, 3, 4, 6, 26])]
    s = "".join(rng.choice(letters) for _ in range(rng.randint(1, 14)))
    check_all(s)
print("12.4 ngẫu nhiên nhỏ: OK")

# 12.5 ngẫu nhiên vừa (so oracle DP) và lớn (so các cách với nhau)
for _ in range(100):
    letters = string.ascii_lowercase[:rng.choice([2, 4, 6, 8])]
    s = "".join(rng.choice(letters) for _ in range(rng.randint(15, 100)))
    check_all(s)
for _ in range(60):
    letters = string.ascii_lowercase[:rng.choice([2, 5, 10, 26])]
    s = "".join(rng.choice(letters) for _ in range(rng.randint(500, 1000)))
    check_all(s, brute=False)
print("12.5 ngẫu nhiên vừa và lớn: OK")

# 12.6 tính chất
for _ in range(500):
    letters = string.ascii_lowercase[:rng.choice([2, 3, 5, 8])]
    s = "".join(rng.choice(letters) for _ in range(rng.randint(1, 30)))
    res = sol(s)
    assert sorted(res) == sorted(set(s)) and len(res) == len(set(s))                 # mỗi ký tự khác nhau đúng một lần
    assert is_subsequence(res, s)                                                    # là dãy con
    assert sol(res) == res                                                           # áp dụng lại: bất động
    assert sol(s[0] * 3 + s) == sol(s)                                               # thêm bản sao ký tự đầu phía trước: không đổi
    if len(set(s)) == len(s):
        assert res == s                                                              # mỗi ký tự một lần: giữ nguyên chuỗi
print("12.6 tính chất: OK")

12.4 ngẫu nhiên nhỏ: OK
12.5 ngẫu nhiên vừa và lớn: OK
12.6 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Sắp xếp các ký tự khác nhau

Đáp án phải là **dãy con** của `s`, nên thứ tự tương đối được giữ; không thể tùy ý sắp xếp lại.

In [22]:
s = counters["sắp xếp các ký tự khác nhau"]
print(f"s={s!r} | sai ->", repr(wrong_sorted_distinct(s)), "| hợp lệ?", is_valid_result(s, wrong_sorted_distinct(s)), "| đúng ->", repr(sol(s)))
assert wrong_sorted_distinct(s) != sol(s) and not is_valid_result(s, wrong_sorted_distinct(s))

s='cba' | sai -> 'abc' | hợp lệ? False | đúng -> 'cba'


Vì sao sai: `"abc"` không là dãy con của `"cba"`. **Đúng:** xây dãy con từng ký tự theo thứ tự xuất hiện (ngăn xếp / tham lam có kiểm tra khả thi).

### Lỗi 2 — Bỏ đỉnh mà không kiểm tra "còn xuất hiện sau"

Bỏ đỉnh lớn hơn `c` mà đỉnh đó không còn xuất hiện nữa làm **mất** ký tự khỏi đáp án.

In [23]:
s = counters["bỏ đỉnh mà không kiểm tra còn xuất hiện sau"]
print(f"s={s!r} | sai ->", repr(wrong_pop_always(s)), "| hợp lệ?", is_valid_result(s, wrong_pop_always(s)), "| đúng ->", repr(sol(s)))
assert wrong_pop_always(s) != sol(s) and not is_valid_result(s, wrong_pop_always(s))

s='ba' | sai -> 'a' | hợp lệ? False | đúng -> 'ba'


Vì sao sai: trong `"ba"`, khi đọc `a`, đỉnh `b > a` nhưng `b` **không còn** xuất hiện sau; bỏ `b` là mất `b` khỏi đáp án (kết quả `"a"` thiếu ký tự). **Đúng:** thêm điều kiện `last[top] > i`.

### Lỗi 3 — Quên bỏ qua ký tự đã có trong ngăn xếp

In [24]:
s = counters["quên bỏ qua ký tự đã có"]
print(f"s={s!r} | sai ->", repr(wrong_no_skip(s)), "| đúng ->", repr(sol(s)))
assert wrong_no_skip(s) != sol(s)

s='cbacdcbc' | sai -> 'acdbc' | đúng -> 'acdb'


Vì sao sai: mỗi ký tự khác nhau phải xuất hiện **đúng một lần**; thêm lại ký tự đã có sinh trùng lặp. **Đúng:** `if c in in_stack: continue`.

### Lỗi 4 — Chỉ giữ lần xuất hiện đầu tiên của mỗi ký tự

In [25]:
s = counters["chỉ giữ lần xuất hiện đầu tiên"]
print(f"s={s!r} | sai ->", repr(wrong_first_occurrence(s)), "| đúng ->", repr(sol(s)))
assert wrong_first_occurrence(s) != sol(s)

s='bcabc' | sai -> 'bca' | đúng -> 'abc'


Vì sao sai: kết quả hợp lệ nhưng không nhỏ nhất, vì lần xuất hiện **sau** của một ký tự đôi khi cho chuỗi nhỏ hơn (ví dụ `"bcabc"`: `"bca"` so với `"abc"`). **Đúng:** cân nhắc cả các lần xuất hiện sau bằng việc bỏ đỉnh khi còn xuất hiện lại.

### Lỗi 5 — Chọn ký tự nhỏ nhất mà không xét khả thi

Chọn ký tự nhỏ nhất còn lại ngay mà không kiểm tra các ký tự còn lại có đủ chỗ phía sau hay không.

In [26]:
s = counters["chọn ký tự nhỏ nhất mà không xét khả thi"]
print(f"s={s!r} | sai ->", repr(wrong_smallest_first(s)), "| hợp lệ?", is_valid_result(s, wrong_smallest_first(s)), "| đúng ->", repr(sol(s)))
assert wrong_smallest_first(s) != sol(s) and not is_valid_result(s, wrong_smallest_first(s))

s='cbacdcbc' | sai -> 'abc' | hợp lệ? False | đúng -> 'acdb'


Vì sao sai: với `"cbacdcbc"`, chọn `a` (vị trí 2) rồi `b` (vị trí 6) bỏ lỡ `d` (vị trí 4). **Đúng:** ký tự được chọn phải **khả thi**: mọi ký tự còn thiếu vẫn xuất hiện từ lần xuất hiện đầu tiên của nó trở đi.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "dãy con nhỏ nhất theo từ điển, mỗi ký tự đúng một lần / xóa `k` chữ số để được số nhỏ nhất" ⇒ **ngăn xếp đơn điệu** với điều kiện "được phép bỏ".
- Điều kiện bỏ đỉnh: đỉnh **lớn hơn** phần tử mới **và** còn cơ hội thêm lại đỉnh sau này (`last[top] > i` hoặc số lần còn lại `> 0`).
- Dùng tập `in_stack` để bỏ qua ký tự đã có.
- Bảng chữ cái nhỏ (`26`) cho `O(26)` bộ nhớ.
- Cùng họ: Remove Duplicate Letters (316, giống hệt), Remove K Digits (402), Next Greater Element, Largest Rectangle in Histogram (84), Daily Temperatures (739).
- Test nên có: một ký tự, toàn một chữ, giảm dần, tăng dần, ký tự cuối nhỏ nhất, ký tự đầu chỉ xuất hiện một lần.

## 15. Biến thể

1. **Dãy con LỚN nhất theo từ điển** chứa mỗi ký tự khác nhau đúng một lần: đổi dấu so sánh `stack[-1] < c`.
2. **Remove Duplicate Letters (316):** đúng bài này.
3. **Remove K Digits (402):** bài tập 5.
4. Bài liên quan: 316, 402, 321 (Create Maximum Number).

Kiểm biến thể 1 (lớn nhất) bằng oracle quy hoạch động (đổi `min` thành `max`):

In [27]:
def largest_stack(s: str) -> str:
    last = {c: i for i, c in enumerate(s)}
    stack, in_stack = [], set()
    for i, c in enumerate(s):
        if c in in_stack:
            continue
        while stack and stack[-1] < c and last[stack[-1]] > i:       # đổi dấu: giữ chuỗi giảm dần càng nhiều càng tốt
            in_stack.remove(stack.pop())
        stack.append(c)
        in_stack.add(c)
    return "".join(stack)

def largest_dp(s: str) -> str:
    letters = sorted(set(s))
    bit = {c: 1 << k for k, c in enumerate(letters)}
    full = (1 << len(letters)) - 1
    n = len(s)

    @lru_cache(maxsize=None)
    def f(i, mask):
        if mask == full:
            return ""
        if i == n:
            return None
        best = f(i + 1, mask)
        c = s[i]
        if not mask & bit[c]:
            rest = f(i + 1, mask | bit[c])
            if rest is not None and (best is None or c + rest > best):
                best = c + rest
        return best

    result = f(0, 0)
    f.cache_clear()
    return result

rng = random.Random(15)
for _ in range(2000):
    letters = string.ascii_lowercase[:rng.choice([2, 3, 4, 6])]
    s = "".join(rng.choice(letters) for _ in range(rng.randint(1, 14)))
    assert largest_stack(s) == largest_dp(s), s
print("biến thể 'lớn nhất': ngăn xếp (đổi dấu) khớp oracle DP; ví dụ:", largest_stack("bcabc"), largest_stack("cbacdcbc"))

biến thể 'lớn nhất': ngăn xếp (đổi dấu) khớp oracle DP; ví dụ: cab cbad


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê dãy con | Theo định nghĩa | `O(2^n n)` | `O(n)` | Oracle, `n <= 16` |
| 2. Chọn tham lam từng ký tự | Ký tự nhỏ nhất khả thi | `O(26^2 n)` | `O(n)` | Dễ chứng minh |
| 3. Ngăn xếp + vị trí cuối | Bỏ đỉnh lớn còn xuất hiện lại | `O(n)` | `O(26)` | **Cách nộp chính** |
| 4. Ngăn xếp + bộ đếm | Giống 3, đếm số lần còn lại | `O(n)` | `O(26)` | Khi không muốn tính `last` trước |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
s = "bcabc"
print({c: i for i, c in enumerate(s)})
print(Counter("abca"))
print(Solution().smallestSubsequence("abcabc"), Solution().smallestSubsequence("cba"))
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # dict vị trí cuối cùng của "bcabc"
my_guess_2 = None      # Counter của "abca"
my_guess_3 = None      # hai kết quả

### Bài 2 — Tính tay (dễ-vừa)

`s = "cdadabcc"`. Ghi `last` của từng ký tự và trạng thái ngăn xếp sau mỗi ký tự; ghi kết quả.

In [29]:
# Bài 2: điền last và kết quả
hand_last, hand_answer = None, None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành ngăn xếp đơn điệu.

In [30]:
def exercise_3(s: str) -> str:
    last = {c: i for i, c in enumerate(s)}
    stack, in_stack = [], set()
    for i, c in enumerate(s):
        # TODO 1: nếu c đã có trong in_stack thì bỏ qua (continue)
        # TODO 2: while stack and stack[-1] > c and last[stack[-1]] > i: bỏ đỉnh (pop) và xóa khỏi in_stack
        # TODO 3: đẩy c vào stack và in_stack
        pass
    return "".join(stack)


# for s, e in examples:
#     assert exercise_3(s) == e

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(s)` bằng **bộ đếm số lần còn lại** (`Counter`).

In [31]:
def exercise_4(s: str) -> str:
    # TODO
    pass


# for s, e in examples:
#     assert exercise_4(s) == e

### Bài 5 — Mini challenge: Remove K Digits (khó)

Cho chuỗi số `num` (các chữ số, không có số `0` đứng đầu nếu `len > 1`) và số nguyên `k`. Xóa đúng `k` chữ số của `num` sao cho số còn lại **nhỏ nhất**; trả về chuỗi kết quả **không có số `0` ở đầu** (nếu rỗng thì trả `"0"`). Ví dụ:

```python
exercise_5("1432219", 3) -> "1219"
exercise_5("10200", 1)   -> "200"
exercise_5("10", 2)      -> "0"
```
Gợi ý: ngăn xếp đơn điệu tăng; bỏ đỉnh lớn hơn chữ số mới khi còn `k > 0`; cuối cùng cắt `k` còn lại ở cuối và bỏ số `0` đầu.

In [32]:
def exercise_5(num: str, k: int) -> str:
    # TODO
    pass


def remove_k_oracle(num, k):
    n = len(num)
    best = None
    for keep in combinations(range(n), n - k):
        digits = "".join(num[i] for i in keep).lstrip("0") or "0"
        key = (len(digits), digits)
        if best is None or key < best[0]:
            best = (key, digits)
    return best[1]


# assert exercise_5("1432219", 3) == "1219"
# assert exercise_5("10200", 1) == "200"
# assert exercise_5("10", 2) == "0"

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `{c: i ...}` ghi đè nên giữ chỉ số **cuối** của mỗi ký tự.
- Mức 2: `"bcabc"`: `b` ở `0` và `3`; `c` ở `1` và `4`; `a` ở `2`.
- Mức 3: `{'b': 3, 'c': 4, 'a': 2}`; `Counter({'a': 2, 'b': 1, 'c': 1})`; `"abc"` và `"cba"`.

**Bài 2**
- Mức 1: `s = "cdadabcc"` (vị trí `0..7`); vị trí cuối: `c` ở `7`, `d` ở `3`, `a` ở `4`, `b` ở `5`.
- Mức 2: `c` → `[c]`; `d` → `[c, d]`; `a` (i = 2): bỏ `d` (`last[d] = 3 > 2`) và bỏ `c` (`last[c] = 7 > 2`), đẩy `a` → `[a]`; `d` (i = 3) → `[a, d]`; `a` (i = 4) bỏ qua; `b` (i = 5): đỉnh `d > b` nhưng `last[d] = 3 < 5` nên giữ → `[a, d, b]`...
- Mức 3: thêm `c` (i = 6) → `[a, d, b, c]`; kết quả `"adbc"`.

**Bài 3**
- Mức 1: `if c in in_stack: continue`.
- Mức 2: `while stack and stack[-1] > c and last[stack[-1]] > i: in_stack.remove(stack.pop())`.
- Mức 3: `stack.append(c)`; `in_stack.add(c)`.

**Bài 4**
- Mức 1: `remain = Counter(s)`.
- Mức 2: mỗi ký tự đầu vòng lặp `remain[c] -= 1`.
- Mức 3: điều kiện bỏ đỉnh: `remain[stack[-1]] > 0`.

**Bài 5**
- Mức 1: ngăn xếp đơn điệu tăng trên các chữ số.
- Mức 2: với mỗi chữ số `d`, `while k and stack and stack[-1] > d: stack.pop(); k -= 1`; rồi `stack.append(d)`.
- Mức 3: kết thúc: `stack = stack[:len(stack) - k]` (cắt `k` còn lại), `"".join(stack).lstrip("0") or "0"`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
s = "bcabc"
r1 = {c: i for i, c in enumerate(s)}
r2 = Counter("abca")
r3 = (Solution().smallestSubsequence("abcabc"), Solution().smallestSubsequence("cba"))
print(r1, r2, r3)
assert r1 == {"b": 3, "c": 4, "a": 2}
assert r2 == Counter({"a": 2, "b": 1, "c": 1})
assert r3 == ("abc", "cba")

{'b': 3, 'c': 4, 'a': 2} Counter({'a': 2, 'b': 1, 'c': 1}) ('abc', 'cba')


### Đáp án Bài 2

`s = "cdadabcc"` (vị trí `0..7`): `last = {c: 7, d: 3, a: 4, b: 5}`.

- `c` → `[c]`; `d` → `[c, d]`.
- `a` (i = 2): đỉnh `d > a` và `last[d] = 3 > 2` ⇒ bỏ `d`; đỉnh `c > a` và `last[c] = 7 > 2` ⇒ bỏ `c`; đẩy `a` ⇒ `[a]`.
- `d` (i = 3): đỉnh `a < d` ⇒ `[a, d]`.
- `a` (i = 4): đã có ⇒ bỏ qua.
- `b` (i = 5): đỉnh `d > b` nhưng `last[d] = 3 < 5` (không còn xuất hiện sau) ⇒ giữ `d`; đẩy `b` ⇒ `[a, d, b]`.
- `c` (i = 6): đỉnh `b < c` ⇒ `[a, d, b, c]`.
- `c` (i = 7): đã có ⇒ bỏ qua.

Kết quả `"adbc"`.

In [34]:
s = "cdadabcc"
hand_last_solution, hand_answer_solution = {"c": 7, "d": 3, "a": 4, "b": 5}, "adbc"
assert {c: i for i, c in enumerate(s)} == hand_last_solution
assert hand_answer_solution == sol(s) == smallest_dp(s) == smallest_brute(s)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(s: str) -> str:
    last = {c: i for i, c in enumerate(s)}
    stack, in_stack = [], set()
    for i, c in enumerate(s):
        if c in in_stack:
            continue
        while stack and stack[-1] > c and last[stack[-1]] > i:
            in_stack.remove(stack.pop())
        stack.append(c)
        in_stack.add(c)
    return "".join(stack)

for s, e in examples:
    assert exercise_3_solution(s) == e
rng = random.Random(3)
for _ in range(500):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 10)))
    assert exercise_3_solution(s) == smallest_brute(s)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(s: str) -> str:
    remain = Counter(s)
    stack, in_stack = [], set()
    for c in s:
        remain[c] -= 1
        if c in in_stack:
            continue
        while stack and stack[-1] > c and remain[stack[-1]] > 0:
            in_stack.remove(stack.pop())
        stack.append(c)
        in_stack.add(c)
    return "".join(stack)

for s, e in examples:
    assert exercise_4_solution(s) == e
rng = random.Random(4)
for _ in range(500):
    s = "".join(rng.choice("abcd") for _ in range(rng.randint(1, 10)))
    assert exercise_4_solution(s) == smallest_brute(s)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(num: str, k: int) -> str:
    stack = []
    for d in num:
        while k and stack and stack[-1] > d:
            stack.pop()
            k -= 1
        stack.append(d)
    stack = stack[:len(stack) - k] if k else stack
    return "".join(stack).lstrip("0") or "0"

def remove_k_oracle_solution(num, k):
    n = len(num)
    best = None
    for keep in combinations(range(n), n - k):
        digits = "".join(num[i] for i in keep).lstrip("0") or "0"
        key = (len(digits), digits)
        if best is None or key < best[0]:
            best = (key, digits)
    return best[1]

assert exercise_5_solution("1432219", 3) == "1219"
assert exercise_5_solution("10200", 1) == "200"
assert exercise_5_solution("10", 2) == "0"
rng = random.Random(5)
for _ in range(500):
    n = rng.randint(1, 9)
    num = str(rng.randint(1, 9)) + "".join(str(rng.randint(0, 9)) for _ in range(n - 1))
    k = rng.randint(0, n)
    assert exercise_5_solution(num, k) == remove_k_oracle_solution(num, k), (num, k)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (ngăn xếp đơn điệu với vị trí cuối cùng)

```python
class Solution:
    def smallestSubsequence(self, s: str) -> str:
        last = {c: i for i, c in enumerate(s)}
        stack, in_stack = [], set()
        for i, c in enumerate(s):
            if c in in_stack:
                continue
            while stack and stack[-1] > c and last[stack[-1]] > i:
                in_stack.remove(stack.pop())
            stack.append(c)
            in_stack.add(c)
        return "".join(stack)
```

Công thức nhớ nhanh: **bỏ qua ký tự đã có; bỏ đỉnh khi `đỉnh > c` và `đỉnh` còn xuất hiện sau; rồi đẩy `c`**.

### Checklist trước khi submit

- [ ] Có `last` (hoặc bộ đếm) để biết đỉnh còn xuất hiện sau hay không?
- [ ] Có tập `in_stack` và bỏ qua ký tự đã có?
- [ ] Khi bỏ đỉnh có xóa nó khỏi `in_stack`?
- [ ] Điều kiện bỏ đỉnh có đủ ba vế: ngăn xếp không rỗng, `đỉnh > c`, đỉnh còn xuất hiện sau?
- [ ] Kết quả là `"".join(stack)` (giữ thứ tự dãy con)?
- [ ] Đã thử: một ký tự, toàn một chữ, giảm dần, tăng dần, `abacb`, `n = 1000`?